# Projeto Integrador — Coleta de Dados da Câmara dos Deputados

**Fonte:** API Dados Abertos da Câmara dos Deputados  
**Base URL:** `https://dadosabertos.camara.leg.br/api/v2/`  
**Legislatura:** 57ª (2023–2026)  

### Tabelas coletadas:
1. `deputados` — dados dos parlamentares
2. `tipos_despesa` — categorias de gasto (tabela de referência)
3. `despesas` — gastos de cada deputado (tabela principal)

---

## Imports

In [ ]:
import requests
import pandas as pd
import time
import json
import os
from datetime import datetime
from google.colab import files

print('Bibliotecas importadas com sucesso!')
print(f'Coleta iniciada em: {datetime.now().strftime("%d/%m/%Y %H:%M:%S")}')

Bibliotecas importadas com sucesso!
Coleta iniciada em: 21/05/2026 19:01:45


## Helper Functions

In [ ]:
def fazer_requisicao(url, params=None, tentativas=3):
    for tentativa in range(1, tentativas + 1):
        try:
            resposta = requests.get(url, params=params, headers=HEADERS, timeout=15)

            if resposta.status_code == 200:
                return resposta.json()
            elif resposta.status_code == 429:
                print(f'Rate limit atingido. Aguardando 5 segundos...')
                time.sleep(5)
            else:
                print(f'Erro HTTP {resposta.status_code} em: {url}')

        except requests.exceptions.Timeout:
            print(f'Timeout na tentativa {tentativa}/{tentativas}')
        except requests.exceptions.ConnectionError:
            print(f'Erro de conexão na tentativa {tentativa}/{tentativas}')
        except Exception as e:
            print(f'Erro inesperado: {e}')

        if tentativa < tentativas:
            time.sleep(2)

    print(f'Falha após {tentativas} tentativas: {url}')
    return None


def coletar_despesas_deputado(id_deputado, anos):
    # Reutilizada na coleta principal e no reprocessamento de falhas
    url = f'{BASE_URL}/deputados/{id_deputado}/despesas'
    todas_despesas = []
    pagina = 1

    while True:
        params = [('pagina', pagina), ('itens', ITENS_POR_PAGINA)]
        for ano in anos:
            params.append(('ano', ano))

        dados = fazer_requisicao(url, params=params)

        if not dados or not dados.get('dados'):
            break

        registros = dados['dados']
        for registro in registros:
            registro['id_deputado'] = id_deputado

        todas_despesas.extend(registros)

        if len(registros) < ITENS_POR_PAGINA:
            break

        pagina += 1
        time.sleep(PAUSA_SEGUNDOS)

    return todas_despesas

print('Funções auxiliares definidas!')

Funções auxiliares definidas!


## Configurações API

In [ ]:
BASE_URL = 'https://dadosabertos.camara.leg.br/api/v2'
ID_LEGISLATURA = 57
ANOS = [2023, 2024, 2025, 2026]
ITENS_POR_PAGINA = 100
PAUSA_SEGUNDOS = 0.3
HEADERS = {'Accept': 'application/json'}
PASTA_DADOS = 'dados_camara'
os.makedirs(PASTA_DADOS, exist_ok=True)

print('Configurações definidas!')
print(f'   Base URL : {BASE_URL}')
print(f'   Legislatura: {ID_LEGISLATURA}')
print(f'   Anos: {ANOS}')
print(f'   Pasta de saída: {PASTA_DADOS}/')

Configurações definidas!
   Base URL : https://dadosabertos.camara.leg.br/api/v2
   Legislatura: 57
   Anos: [2023, 2024, 2025, 2026]
   Pasta de saída: dados_camara/


## Coleta dos dados

Tabela Tipos de Despesa (referência)

In [ ]:
print('Coletando tipos de despesa...')

url = f'{BASE_URL}/referencias/deputados/tipoDespesa'
dados = fazer_requisicao(url)

if dados and 'dados' in dados:
    df_tipos_despesa = pd.DataFrame(dados['dados'])
    print(f'{len(df_tipos_despesa)} tipos de despesa coletados!')
else:
    print('Falha ao coletar tipos de despesa.')
    df_tipos_despesa = pd.DataFrame()

display(df_tipos_despesa)

Coletando tipos de despesa...
23 tipos de despesa coletados!


,cod,sigla,nome,descricao
0,1,,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,
1,2,,"LOCOMOÇÃO, ALIMENTAÇÃO E HOSPEDAGEM",
2,3,,COMBUSTÍVEIS E LUBRIFICANTES.,
3,4,,"CONSULTORIAS, PESQUISAS E TRABALHOS TÉCNICOS.",
4,5,,DIVULGAÇÃO DA ATIVIDADE PARLAMENTAR.,
5,6,,AQUISIÇÃO DE MATERIAL DE ESCRITÓRIO.,
6,7,,AQUISIÇÃO OU LOC. DE SOFTWARE; SERV. POSTAIS; ...,
7,8,,SERVIÇO DE SEGURANÇA PRESTADO POR EMPRESA ESPE...,
8,9,,PASSAGEM AÉREA - REEMBOLSO,
9,10,,TELEFONIA,


Tabela Deputados

In [ ]:
print(f'Coletando deputados da {ID_LEGISLATURA}ª Legislatura...')

url = f'{BASE_URL}/deputados'
todos_deputados = []
pagina = 1

while True:
    params = {
        'idLegislatura': ID_LEGISLATURA,
        'ordem': 'ASC',
        'ordenarPor': 'nome',
        'pagina': pagina,
        'itens': ITENS_POR_PAGINA
    }

    dados = fazer_requisicao(url, params=params)

    if not dados or not dados.get('dados'):
        break

    registros = dados['dados']
    todos_deputados.extend(registros)
    print(f'  Página {pagina}: {len(registros)} deputados | Total: {len(todos_deputados)}')

    if len(registros) < ITENS_POR_PAGINA:
        break

    pagina += 1
    time.sleep(PAUSA_SEGUNDOS)

df_deputados = pd.DataFrame(todos_deputados)

colunas_uteis = {'id': 'id_deputado', 'nome': 'nome', 'siglaPartido': 'partido', 'siglaUf': 'uf', 'uri': 'uri'}
colunas_existentes = {k: v for k, v in colunas_uteis.items() if k in df_deputados.columns}
df_deputados = df_deputados[list(colunas_existentes.keys())].rename(columns=colunas_existentes)

print(f'\nTotal de deputados coletados: {len(df_deputados)}')
display(df_deputados.head(10))

Coletando deputados da 57ª Legislatura...
  Página 1: 100 deputados | Total: 100
  Página 2: 100 deputados | Total: 200
  Página 3: 100 deputados | Total: 300
  Página 4: 100 deputados | Total: 400
  Página 5: 100 deputados | Total: 500
  Página 6: 100 deputados | Total: 600
  Página 7: 100 deputados | Total: 700
  Página 8: 100 deputados | Total: 800
  Página 9: 67 deputados | Total: 867

Total de deputados coletados: 867


,id_deputado,nome,partido,uf,uri
0,220593,Abilio Brunini,PL,MT,https://dadosabertos.camara.leg.br/api/v2/depu...
1,204379,Acácio Favacho,MDB,AP,https://dadosabertos.camara.leg.br/api/v2/depu...
2,220714,Adail Filho,REPUBLICANOS,AM,https://dadosabertos.camara.leg.br/api/v2/depu...
3,220714,Adail Filho,MDB,AM,https://dadosabertos.camara.leg.br/api/v2/depu...
4,221328,Adilson Barroso,PL,SP,https://dadosabertos.camara.leg.br/api/v2/depu...
5,204560,Adolfo Viana,PSDB,BA,https://dadosabertos.camara.leg.br/api/v2/depu...
6,204528,Adriana Ventura,NOVO,SP,https://dadosabertos.camara.leg.br/api/v2/depu...
7,121948,Adriano do Baldy,PP,GO,https://dadosabertos.camara.leg.br/api/v2/depu...
8,74646,Aécio Neves,PSDB,MG,https://dadosabertos.camara.leg.br/api/v2/depu...
9,160508,Afonso Florence,PT,BA,https://dadosabertos.camara.leg.br/api/v2/depu...


Tabela Despesas (tabela principal)

In [ ]:
total_deputados = len(df_deputados)
todas_despesas = []
falhas = []

print(f'Iniciando coleta de despesas para {total_deputados} deputados...')
print(f'   Anos: {ANOS}')
print(f'   Estimativa: pode levar entre 20 e 40 minutos\n')

for i, row in df_deputados.iterrows():
    id_dep = row['id_deputado']
    nome_dep = row['nome']
    numero = df_deputados.index.get_loc(i) + 1

    if numero % 10 == 0 or numero == 1:
        print(f'  [{numero}/{total_deputados}] Coletando: {nome_dep} (ID: {id_dep})')

    despesas = coletar_despesas_deputado(id_dep, ANOS)

    if despesas:
        todas_despesas.extend(despesas)
    else:
        falhas.append(id_dep)

    if numero % 50 == 0:
        df_parcial = pd.DataFrame(todas_despesas)
        df_parcial.to_parquet(f'{PASTA_DADOS}/despesas_checkpoint_{numero}.parquet', index=False)
        print(f'  Checkpoint salvo: {len(todas_despesas)} registros acumulados')

    time.sleep(PAUSA_SEGUNDOS)

df_despesas_raw = pd.DataFrame(todas_despesas)

print(f'\nColeta finalizada!')
print(f'   Total de registros: {len(df_despesas_raw)}')
print(f'   Deputados com falha: {len(falhas)} {falhas if falhas else ""}')

Iniciando coleta de despesas para 867 deputados...
   Anos: [2023, 2024, 2025, 2026]
   Estimativa: pode levar entre 20 e 40 minutos

  [1/867] Coletando: Abilio Brunini (ID: 220593)
  [10/867] Coletando: Afonso Florence (ID: 160508)
  [20/867] Coletando: Alencar Santana (ID: 204501)
  [30/867] Coletando: Alfredo Gaspar (ID: 220576)
  [40/867] Coletando: Amanda Gentil (ID: 220707)
  [50/867] Coletando: André Ferreira (ID: 204423)
  Checkpoint salvo: 59744 registros acumulados
  [60/867] Coletando: Antonio Andrade (ID: 220544)
  [70/867] Coletando: Arthur Oliveira Maia (ID: 160600)
  [80/867] Coletando: Bandeira de Mello (ID: 220605)
  [90/867] Coletando: Bibo Nunes (ID: 204388)
  [100/867] Coletando: Capitão Alden (ID: 220690)
  Checkpoint salvo: 120226 registros acumulados
  [110/867] Coletando: Carlos Jordy (ID: 204460)
  [120/867] Coletando: Célia Xakriabá (ID: 206018)
  [130/867] Coletando: Chico Alencar (ID: 74171)
  [140/867] Coletando: Coronel Armando (ID: 204376)
  [150/867] Co

### Reprocessamento das falhas

In [ ]:
print(f'Verificando {len(falhas)} deputados com falha...\n')

registros_recuperados = []
diagnostico = []

for id_dep in falhas:
    url = f'{BASE_URL}/deputados/{id_dep}/despesas'
    params = [('pagina', 1), ('itens', 10)]
    for ano in ANOS:
        params.append(('ano', ano))

    try:
        resposta = requests.get(url, params=params, headers=HEADERS, timeout=15)
        status = resposta.status_code
        print(f'ID {id_dep} -> status {status}')

        if status == 204:
            motivo = 'sem despesas no periodo'
        elif status == 404:
            motivo = 'deputado nao encontrado'
        elif status == 200:
            motivo = 'conexao ok, tentando novamente'
            time.sleep(5)
            despesas = coletar_despesas_deputado(id_dep, ANOS)
            if despesas:
                registros_recuperados.extend(despesas)
                motivo = f'recuperado: {len(despesas)} registros'
        else:
            motivo = f'erro http {status}'

    except Exception as e:
        status = None
        motivo = f'erro de conexao: {e}'

    diagnostico.append({'id_deputado': id_dep, 'status': status, 'resultado': motivo})
    print(f'   -> {motivo}')

df_diagnostico = pd.DataFrame(diagnostico)
display(df_diagnostico)

Verificando 6 deputados com falha...

ID 204503 -> status 200
   -> conexao ok, tentando novamente
ID 236284 -> status 200
   -> conexao ok, tentando novamente
ID 74295 -> status 200
   -> conexao ok, tentando novamente
ID 235869 -> status 200
   -> conexao ok, tentando novamente
ID 220636 -> status 200
   -> conexao ok, tentando novamente
ID 220643 -> status 200
   -> conexao ok, tentando novamente


,id_deputado,status,resultado
0,204503,200,"conexao ok, tentando novamente"
1,236284,200,"conexao ok, tentando novamente"
2,74295,200,"conexao ok, tentando novamente"
3,235869,200,"conexao ok, tentando novamente"
4,220636,200,"conexao ok, tentando novamente"
5,220643,200,"conexao ok, tentando novamente"


In [ ]:
if registros_recuperados:
    df_extra = pd.DataFrame(registros_recuperados)
    df_despesas_raw = pd.concat([df_despesas_raw, df_extra], ignore_index=True)
    print(f'Registros recuperados adicionados. Total agora: {len(df_despesas_raw)}')
else:
    print('Nenhum registro recuperado. As falhas eram ausencias legitimas.')

df_diagnostico.to_csv(f'{PASTA_DADOS}/diagnostico_falhas.csv', index=False, encoding='utf-8-sig')
print('Diagnostico salvo.')

Nenhum registro recuperado. As falhas eram ausencias legitimas.
Diagnostico salvo.


## Tratamento inicial dos dados

Tabela Despesas

In [ ]:
print('Iniciando tratamento das despesas...')
print(f'   Registros brutos: {len(df_despesas_raw)}')

df_despesas = df_despesas_raw.copy()

# 1. Seleciona e renomeia colunas relevantes
mapa_colunas = {
    'id_deputado'       : 'id_deputado',
    'ano'               : 'ano',
    'mes'               : 'mes',
    'tipoDespesa'       : 'tipo_despesa',
    'codDocumento'      : 'cod_documento',
    'dataDocumento'     : 'data_documento',
    'valorDocumento'    : 'valor_documento',
    'valorLiquido'      : 'valor_liquido',
    'valorGlosa'        : 'valor_glosa',
    'nomeFornecedor'    : 'nome_fornecedor',
    'cnpjCpfFornecedor' : 'cnpj_cpf_fornecedor',
    'numDocumento'      : 'num_documento',
    'urlDocumento'      : 'url_documento'
}
colunas_presentes = {k: v for k, v in mapa_colunas.items() if k in df_despesas.columns}
df_despesas = df_despesas[list(colunas_presentes.keys())].rename(columns=colunas_presentes)

# 2. Conversão de tipos
df_despesas['data_documento'] = pd.to_datetime(df_despesas['data_documento'], errors='coerce')

for col in ['valor_documento', 'valor_liquido', 'valor_glosa']:
    if col in df_despesas.columns:
        df_despesas[col] = pd.to_numeric(df_despesas[col], errors='coerce')

for col in ['ano', 'mes']:
    if col in df_despesas.columns:
        df_despesas[col] = pd.to_numeric(df_despesas[col], errors='coerce').astype('Int64')

# Identificadores como texto — não são valores numéricos
for col in ['id_deputado', 'cod_documento', 'cnpj_cpf_fornecedor', 'num_documento']:
    if col in df_despesas.columns:
        df_despesas[col] = df_despesas[col].astype(str).str.strip()

# 3. Trata nulos
df_despesas['nome_fornecedor'] = df_despesas['nome_fornecedor'].fillna('NAO INFORMADO')
df_despesas['cnpj_cpf_fornecedor'] = df_despesas['cnpj_cpf_fornecedor'].fillna('NAO INFORMADO')
df_despesas = df_despesas.dropna(subset=['valor_liquido'])

# 4. Remove duplicatas
linhas_antes = len(df_despesas)
df_despesas = df_despesas.drop_duplicates(subset=['cod_documento', 'id_deputado'])
duplicatas_removidas = linhas_antes - len(df_despesas)

print(f'   Duplicatas removidas: {duplicatas_removidas}')
print(f'   Registros finais: {len(df_despesas)}')
print(f'\nNulos restantes:')
print(df_despesas.isnull().sum()[df_despesas.isnull().sum() > 0])

display(df_despesas.head())
print(df_despesas.dtypes)

Iniciando tratamento das despesas...
   Registros brutos: 973640
   Duplicatas removidas: 304729
   Registros finais: 668911

Nulos restantes:
url_documento    156415
dtype: int64


,id_deputado,ano,mes,tipo_despesa,cod_documento,data_documento,valor_documento,valor_liquido,valor_glosa,nome_fornecedor,cnpj_cpf_fornecedor,num_documento,url_documento
0,220593,2023,9,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,7610066,2023-09-19,52.98,52.98,0.0,3XIS COMERCIO VAREJISTA E ATACADISTA DE PAPELA...,42999796000188,35537,http://www.camara.leg.br/cota-parlamentar/nota...
1,220593,2023,11,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,7657491,2023-11-27,67.30,67.30,0.0,AGUAS CUIABA S.A,14995581000153,01,https://www.camara.leg.br/cota-parlamentar/doc...
2,220593,2023,5,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,7552621,2023-05-05,43.20,43.20,0.0,AGUAS CUIABA S.A,14995581000153,11533052023001,https://www.camara.leg.br/cota-parlamentar/doc...
3,220593,2023,6,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,7587236,2023-06-05,43.20,43.20,0.0,AGUAS CUIABA S.A,14995581000153,11533062023001,https://www.camara.leg.br/cota-parlamentar/doc...
4,220593,2023,7,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,7587239,2023-07-05,43.20,43.20,0.0,AGUAS CUIABA S.A,14995581000153,11533072023001,https://www.camara.leg.br/cota-parlamentar/doc...


id_deputado                    object
ano                             Int64
mes                             Int64
tipo_despesa                   object
cod_documento                  object
data_documento         datetime64[ns]
valor_documento               float64
valor_liquido                 float64
valor_glosa                   float64
nome_fornecedor                object
cnpj_cpf_fornecedor            object
num_documento                  object
url_documento                  object
dtype: object


Tabela Deputados

In [ ]:
df_deputados['id_deputado'] = df_deputados['id_deputado'].astype(str)
df_deputados['partido'] = df_deputados['partido'].fillna('SEM PARTIDO')
df_deputados['uf'] = df_deputados['uf'].fillna('NAO INFORMADO')
df_deputados = df_deputados.drop_duplicates(subset=['id_deputado'])

print(f'Deputados após tratamento: {len(df_deputados)}')
print(f'Nulos restantes:\n{df_deputados.isnull().sum()}')

Deputados após tratamento: 641
Nulos restantes:
id_deputado    0
nome           0
partido        0
uf             0
uri            0
dtype: int64


Tabela tipo despesas

In [ ]:
shape = df_tipos_despesa.shape
tipos = df_tipos_despesa.dtypes
nulos = df_tipos_despesa.isna().sum()
duplicatas = df_tipos_despesa.duplicated().sum()

print(f'Registros: {shape[0]} linhas, {shape[1]} colunas')
print(f'\nTipos de dados:')
print(tipos.to_string())
print(f'\nNulos por coluna:')
print(nulos.to_string())
print(f'\nDuplicatas: {duplicatas}')

Registros: 23 linhas, 4 colunas

Tipos de dados:
cod          object
sigla        object
nome         object
descricao    object

Nulos por coluna:
cod          0
sigla        0
nome         0
descricao    0

Duplicatas: 0


## Resumo da coleta

In [ ]:
print('=' * 55)
print('RESUMO DA COLETA')
print('=' * 55)

print(f'\nDEPUTADOS')
print(f'   Total: {len(df_deputados)}')
print(f'   Partidos diferentes: {df_deputados["partido"].nunique()}')
print(f'   Estados diferentes : {df_deputados["uf"].nunique()}')

print(f'\nTIPOS DE DESPESA')
print(f'   Total de categorias: {len(df_tipos_despesa)}')

print(f'\nDESPESAS')
print(f'   Total de registros : {len(df_despesas)}')
print(f'   Período coletado   : {df_despesas["ano"].min()} a {df_despesas["ano"].max()}')
print(f'   Valor total (R$)   : {df_despesas["valor_liquido"].sum():,.2f}')
print(f'   Valor médio (R$)   : {df_despesas["valor_liquido"].mean():,.2f}')
print(f'   Valor mediano (R$) : {df_despesas["valor_liquido"].median():,.2f}')
print(f'   Valor mínimo (R$)  : {df_despesas["valor_liquido"].min():,.2f}')
print(f'   Valor máximo (R$)  : {df_despesas["valor_liquido"].max():,.2f}')

print(f'\nTOP 5 TIPOS DE DESPESA POR VALOR TOTAL:')
top5 = (df_despesas
        .groupby('tipo_despesa')['valor_liquido']
        .sum()
        .sort_values(ascending=False)
        .head(5)
        .reset_index())
top5.columns = ['Tipo de Despesa', 'Valor Total (R$)']
top5['Valor Total (R$)'] = top5['Valor Total (R$)'].map('{:,.2f}'.format)
display(top5)

print('=' * 55)

RESUMO DA COLETA

DEPUTADOS
   Total: 641
   Partidos diferentes: 23
   Estados diferentes : 27

TIPOS DE DESPESA
   Total de categorias: 23

DESPESAS
   Total de registros : 668911
   Período coletado   : 2023 a 2026
   Valor total (R$)   : 782,120,689.69
   Valor médio (R$)   : 1,169.24
   Valor mediano (R$) : 250.06
   Valor mínimo (R$)  : -6,683.44
   Valor máximo (R$)  : 184,428.00

TOP 5 TIPOS DE DESPESA POR VALOR TOTAL:


,Tipo de Despesa,Valor Total (R$)
0,DIVULGAÇÃO DA ATIVIDADE PARLAMENTAR.,"328,375,273.15"
1,LOCAÇÃO OU FRETAMENTO DE VEÍCULOS AUTOMOTORES,"134,644,932.53"
2,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,"104,920,156.58"
3,PASSAGEM AÉREA - SIGEPA,"96,508,809.49"
4,COMBUSTÍVEIS E LUBRIFICANTES.,"72,036,100.25"


## Salvar as tabelas

In [ ]:
print('Salvando tabelas...')

df_deputados.to_parquet(f'{PASTA_DADOS}/deputados.parquet', index=False)
df_tipos_despesa.to_parquet(f'{PASTA_DADOS}/tipos_despesa.parquet', index=False)
df_despesas.to_parquet(f'{PASTA_DADOS}/despesas.parquet', index=False)

df_deputados.to_csv(f'{PASTA_DADOS}/deputados.csv', index=False, encoding='utf-8-sig')
df_tipos_despesa.to_csv(f'{PASTA_DADOS}/tipos_despesa.csv', index=False, encoding='utf-8-sig')
df_despesas.to_csv(f'{PASTA_DADOS}/despesas.csv', index=False, encoding='utf-8-sig')

print('Arquivos salvos em:', PASTA_DADOS)
for arquivo in os.listdir(PASTA_DADOS):
    caminho = os.path.join(PASTA_DADOS, arquivo)
    tamanho_mb = os.path.getsize(caminho) / (1024 * 1024)
    print(f'   {arquivo}: {tamanho_mb:.2f} MB')

Salvando tabelas...
Arquivos salvos em: dados_camara
   deputados.csv: 0.05 MB
   despesas_checkpoint_250.parquet: 10.27 MB
   despesas_checkpoint_350.parquet: 13.47 MB
   despesas_checkpoint_750.parquet: 27.68 MB
   despesas_checkpoint_800.parquet: 29.52 MB
   deputados.parquet: 0.02 MB
   despesas_checkpoint_100.parquet: 4.41 MB
   despesas_checkpoint_700.parquet: 25.78 MB
   diagnostico_falhas.csv: 0.00 MB
   despesas.csv: 134.37 MB
   despesas_checkpoint_550.parquet: 20.05 MB
   despesas_checkpoint_850.parquet: 31.59 MB
   despesas_checkpoint_400.parquet: 15.07 MB
   despesas_checkpoint_500.parquet: 18.22 MB
   despesas.parquet: 23.38 MB
   despesas_checkpoint_50.parquet: 2.35 MB
   despesas_checkpoint_650.parquet: 23.83 MB
   despesas_checkpoint_450.parquet: 16.85 MB
   despesas_checkpoint_150.parquet: 6.39 MB
   tipos_despesa.csv: 0.00 MB
   despesas_checkpoint_300.parquet: 11.63 MB
   despesas_checkpoint_600.parquet: 21.87 MB
   despesas_checkpoint_200.parquet: 8.36 MB
   tipos_

## Download dos arquivos

In [ ]:
files.download(f'{PASTA_DADOS}/deputados.csv')
files.download(f'{PASTA_DADOS}/tipos_despesa.csv')
files.download(f'{PASTA_DADOS}/despesas.csv')

print('Downloads iniciados!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads iniciados!


## Registro da coleta

In [ ]:
registro = {
    'data_coleta'             : datetime.now().strftime('%d/%m/%Y %H:%M:%S'),
    'api'                     : 'Dados Abertos da Câmara dos Deputados',
    'legislatura'             : ID_LEGISLATURA,
    'anos_coletados'          : ANOS,
    'total_deputados'         : len(df_deputados),
    'total_tipos_despesa'     : len(df_tipos_despesa),
    'total_registros_despesa' : len(df_despesas),
    'valor_total_coletado'    : float(df_despesas['valor_liquido'].sum())
}

with open(f'{PASTA_DADOS}/registro_coleta.json', 'w', encoding='utf-8') as f:
    json.dump(registro, f, ensure_ascii=False, indent=2)

print('Registro da coleta:')
for chave, valor in registro.items():
    print(f'   {chave}: {valor}')

Registro da coleta:
   data_coleta: 21/05/2026 22:28:57
   api: Dados Abertos da Câmara dos Deputados
   legislatura: 57
   anos_coletados: [2023, 2024, 2025, 2026]
   total_deputados: 641
   total_tipos_despesa: 23
   total_registros_despesa: 668911
   valor_total_coletado: 782120689.6900002
